# Lab 15: The Click Threshold

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-15.ipynb)

**What you will do:** find your own auditory click-fusion and visual flicker-fusion thresholds in hertz, compare them with the typical ranges Lab 15 reports, and watch a simulated camera alias a fast-spinning wheel once its frame rate falls below the wheel's true rotation rate.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Flashing lights carry a small photosensitivity risk for people with a personal or family
history of photosensitive epilepsy. Step 3 asks you to ramp the flash frequency up gradually,
which means briefly passing through 15-25 Hz, the range most likely to trigger a seizure in
susceptible people. Stop immediately if you feel unwell, and skip the visual flicker step
altogether if you or anyone watching has any history of light-triggered seizures.

## The experiment in the book

In Lab 15 you raised the speed of a metronome, then a click generator, until individual
clicks blurred into a buzz, somewhere around 15-20 clicks per second to stop counting them,
and about 50 Hz for them to fuse completely. You then found the same kind of threshold for
flashing light, around 50-60 Hz. Both thresholds mark the **temporal resolution** of a sense:
the finest rate of separate events the brain can still tell apart, set by how quickly its
neurons can fire and recover.

## Record your results

Record the frequency (in Hz) at which clicks blurred into a buzz, the frequency at which
they fully fused into a continuous tone, and the frequency at which flashes fused into
steady light. If your apps report beats per minute (bpm), divide by 60 to get Hz.

In [ ]:
# example data: replace with your own thresholds, in Hz
thresholds = pd.Series({
    "clicks hard to count": 17.0,
    "clicks fully fused (tone)": 48.0,
    "flicker fully fused (light)": 55.0,
})
display(thresholds.to_frame("Hz"))
thresholds.plot.barh(color=["steelblue", "steelblue", "firebrick"], figsize=(6, 2.5))
plt.xlabel("Hz"); plt.gca().invert_yaxis(); plt.show()

## Compare

Lab 15 reports that clicks become hard to count "somewhere around 15-20 Hz," fuse
completely "by about 50 Hz," and that visual flicker fusion is "typically around 50-60 Hz."
Compare your three thresholds with these ranges.

In [ ]:
ranges = {
    "clicks hard to count": (15, 20),
    "clicks fully fused (tone)": (45, 55),   # "by about 50 Hz" -- a narrow band around the stated value
    "flicker fully fused (light)": (50, 60),
}
for label, value in thresholds.items():
    lo, hi = ranges[label]
    status = "inside" if lo <= value <= hi else ("below" if value < lo else "above")
    print(f"{label}: {value:.0f} Hz -- {status} the range the lab reports ({lo}-{hi} Hz)")
print()
print("App-reported thresholds depend a lot on the specific app and your listening/viewing conditions,")
print("so treat 'outside the range' as a prompt to retest, not a sign that something is wrong with you.")

## The AI side

Lab 15's AI Connection explains the **wagon-wheel illusion**: a camera samples a rotating
wheel at a fixed frame rate, and if the wheel turns fast enough, each frame catches it at
nearly the same angle as the previous one turned by a bit more than a full rotation, so it
looks like it is turning slowly, standing still, or spinning backward. This is genuine
**aliasing** in the camera's sampling process, distinct from the clicks blurring into a buzz
you just measured, which the lab describes as temporal integration rather than aliasing. The
code below "films" a spinning wheel at different frame rates and shows what each one aliases
the true rotation speed into.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np

true_rps = 10.0          # true rotation speed: 10 revolutions per second
frame_rates = [5, 8, 9, 10, 12, 24, 30]   # frames per second, like different cameras

rows = []
for fps in frame_rates:
    # angle the wheel has turned between consecutive frames, in revolutions
    turn_per_frame = true_rps / fps
    # the visual system (or a naive frame-differencer) reports the smallest turn
    # consistent with what it sees: wrap into (-0.5, 0.5] revolutions per frame
    aliased_turn = (turn_per_frame + 0.5) % 1.0 - 0.5
    apparent_rps = aliased_turn * fps
    rows.append({"frame rate (fps)": fps, "true rps": true_rps,
                 "apparent rps": round(apparent_rps, 2),
                 "looks like": ("forward" if apparent_rps > 0.05 else
                                "backward" if apparent_rps < -0.05 else "stopped")})
aliasing = pd.DataFrame(rows)
display(aliasing)

plt.figure(figsize=(6, 3.5))
plt.axhline(0, color="grey", linewidth=0.8)
plt.plot(aliasing["frame rate (fps)"], aliasing["apparent rps"], "o-", color="steelblue")
plt.axhline(true_rps, color="firebrick", linestyle="--", label="true speed (10 rps)")
plt.xlabel("frame rate (fps)"); plt.ylabel("apparent speed (rps)"); plt.legend(); plt.show()

Notice that only frame rates well above 20 fps (twice the true 10 rps, the Nyquist limit
named in the text) show the wheel spinning forward at close to its true speed; slower frame
rates alias it into a slower, stationary, or backward-spinning wheel. The disanalogy: a
camera's frame rate is a hard, fixed sampling interval, while your own visual system's
"sampling" is not a single clean rate but the outcome of many overlapping, noisier neural
processes, which is why the wagon-wheel illusion can also appear under continuous,
non-flickering daylight with no camera or strobing light involved at all, as the text notes.

## Pool the class data

Pool the class's click and flicker thresholds to see the spread across people. Each row is
one anonymous ID and their three thresholds, in Hz.

In [ ]:
import io
csv_text = """id,clicks_hard_hz,clicks_fused_hz,flicker_fused_hz
P01,16,46,52
P02,19,51,58
P03,15,44,50
P04,20,53,60
P05,18,49,55
"""  # example data: replace with your class CSV, e.g. pd.read_csv("thresholds_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
long = class_df.melt(id_vars="id", var_name="measure", value_name="hz")

summary = long.groupby("measure")["hz"].agg(["mean", "std"])
display(summary)

boot_summary = {}
for measure, group in long.groupby("measure"):
    boot = [group["hz"].sample(len(group), replace=True).mean() for _ in range(3000)]
    boot_summary[measure] = np.percentile(boot, [2.5, 97.5])
for measure, (lo, hi) in boot_summary.items():
    print(f"{measure}: mean {summary.loc[measure, 'mean']:.1f} Hz (95% bootstrap interval {lo:.1f}-{hi:.1f} Hz)")

long.boxplot(column="hz", by="measure", grid=False, figsize=(6, 4))
plt.suptitle(""); plt.title("Class thresholds"); plt.ylabel("Hz"); plt.xticks(rotation=20); plt.show()

## Questions to think about

1. Your auditory "clicks hard to count" threshold (around 15-20 Hz) is much lower than the finest gap the auditory system can in principle detect (2-5 ms, i.e. up to several hundred Hz for a single gap). What does the text say is the difference between detecting a single gap and following a whole train of repeating clicks?
2. Why does the wagon-wheel simulation alias to a "backward" apparent speed for some frame rates but "forward" for others, even though the true rotation direction never changes? Look at how the aliased_turn calculation wraps the value.
3. The temporal resolution of hearing (about 2 ms) is roughly ten times finer than vision (about 20-30 ms). Lab 15 links this to the ventriloquism effect in Lab 16, where hearing dominates judgements of when and vision dominates judgements of where. Why would a sense with finer timing resolution be better suited to answering "when" questions?
4. CD audio samples at 44,100 Hz to capture frequencies up to about 22 kHz, following the Nyquist theorem. Using the same logic, what is the slowest frame rate that could, in principle, capture a wheel spinning at 10 revolutions per second without aliasing?

## Challenge

Test whether the auditory advantage the text describes, detecting tiny gaps between just two
sounds, as opposed to counting a train of repeating clicks, actually feels different to you.
Play two clicks with a large gap (say, 200 ms) and shrink the gap in steps until you can no
longer tell there were two clicks rather than one. Log the smallest gap (in ms) at which you
could still hear two separate sounds, and compare it with the 50-70 ms spacing at which a
repeating click train blurred into a buzz in Step 2.

In [ ]:
# Example data: replace with your own results from the two-click gap-detection test.
two_click_gaps_ms = [200, 100, 50, 25, 15, 10, 5]            # example data: gaps you tried, in ms
heard_as_two = [True, True, True, True, True, False, False]  # example data: could you still hear two clicks?

result = pd.DataFrame({"gap (ms)": two_click_gaps_ms, "heard as two clicks": heard_as_two})
display(result)
smallest_two_click_gap = min(g for g, heard in zip(two_click_gaps_ms, heard_as_two) if heard)
print(f"Smallest gap you could still hear as two separate clicks: {smallest_two_click_gap} ms")
print("Compare this with the repeating click train in Step 2, which blurred into a buzz somewhere around")
print("50-70 ms between clicks (15-20 per second). Lab 15's point is that these are different tasks, so a")
print("much smaller number here is expected, not a contradiction.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-15.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")